# KEEPER
## Urban Investment & Resilience Intelligence

This notebook demonstrates how KEEPER identifies priority flood-resilience investments for Lahore under a constrained budget.

**Pipeline:** CSV -> Risk Engine -> Benefit Engine -> OR-Tools Optimizer -> Investment Plan -> Map / Policy Brief

> Run **Runtime -> Run all**. This notebook clones KEEPER from GitHub, installs dependencies, and runs the full engine end-to-end.


## 01 — Setup: clone repo & install dependencies

In [ ]:
# If running in Colab, uncomment and set your repo URL:
# !git clone https://github.com/<your-username>/keeper.git
# %cd keeper

!pip install -q ortools pandas folium gradio plotly fpdf2 geopandas


## 02 — Load Data
Zones, population, risk variables, and interventions.

In [ ]:
import sys
sys.path.insert(0, 'src')

import pandas as pd
from risk_engine import run_risk_engine, load_weights
from benefit_engine import run_benefit_engine, load_interventions
from optimizer import optimize
from scenario import Scenario
from policy import generate_policy_brief

scenario = Scenario()
zones = pd.read_csv(scenario.zones_file)
interventions = load_interventions()

print('Zones:', len(zones))
print('Interventions:', len(interventions))
zones.head()


## 03 — Risk Analysis
Zone -> Risk Score, then a chart.

In [ ]:
scored = run_risk_engine(scenario.zones_file, scenario.scenario_name)
scored_sorted = scored.sort_values('flood_risk', ascending=False)
scored_sorted[['zone_id', 'zone_name', 'flood_risk', 'risk_class']].head(10)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
colors = scored_sorted['risk_class'].map({'LOW': 'green', 'MODERATE': 'orange', 'HIGH': 'red', 'CRITICAL': 'darkred'})
ax.barh(scored_sorted['zone_name'], scored_sorted['flood_risk'], color=colors)
ax.set_xlabel('Flood Risk (0-100)')
ax.set_title('KEEPER — Flood Risk by Zone (Lahore, demonstration data)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 04 — Risk Map
Display a map of the demonstration zones.

In [ ]:
import folium

center = scenario.city['center']
m = folium.Map(location=[center['lat'], center['lon']], zoom_start=scenario.city['default_zoom'])
colors = {'LOW': 'green', 'MODERATE': 'orange', 'HIGH': 'red', 'CRITICAL': 'darkred'}

for row in scored.itertuples():
    folium.CircleMarker(
        location=[row.lat, row.lon], radius=7,
        color=colors[row.risk_class], fill=True, fill_color=colors[row.risk_class], fill_opacity=0.8,
        popup=f'{row.zone_name} ({row.zone_id}) — {row.flood_risk}/100 [{row.risk_class}]',
    ).add_to(m)

m


## 05 — Intervention Analysis
Zone x Intervention -> Cost, Expected Benefit.

In [ ]:
benefits = run_benefit_engine(scored)
benefits[['zone_name', 'intervention_name', 'cost', 'benefit_score', 'expected_population_helped']].head(15)


## 06 — Optimization
Set a budget and run the OR-Tools optimizer.

In [ ]:
budget = 100_000_000  # PKR
result = optimize(benefits, budget, objective='maximum_population_benefit', objectives_config=scenario.objectives)
result


## 07 — Results
Recommended Portfolio, Total Cost, Remaining Budget, Population Benefit, Risk Reduction.

In [ ]:
print(f"Total Cost: PKR {result['total_cost']:,.0f}")
print(f"Remaining Budget: PKR {result['remaining_budget']:,.0f}")
print(f"Population Benefit: {result['population_benefit']:,.0f}")
print(f"Risk Reduction: {result['risk_reduction']}%")

pd.DataFrame(result['recommendations'])


## 08 — What-if
Change the budget and run again. The recommended plan changes — this proves the engine isn't a static mockup.

In [ ]:
budget_low = 50_000_000
result_low = optimize(benefits, budget_low, objective='maximum_population_benefit', objectives_config=scenario.objectives)

print(f"--- Budget PKR {budget_low:,.0f} ---")
print(f"Total Cost: PKR {result_low['total_cost']:,.0f}")
print(f"Population Benefit: {result_low['population_benefit']:,.0f}")
print(f"Risk Reduction: {result_low['risk_reduction']}%")
pd.DataFrame(result_low['recommendations'])


## 09 — AI Explanation (optional)
The LLM never calculates anything — it only explains a result that OR-Tools already computed.

Configure `AI_PROVIDER` and an API key as an environment variable / Colab secret if you want live explanations; KEEPER works fully without this step.

In [ ]:
explanation_payload = {
    'budget': budget,
    'selected_interventions': result['recommendations'],
    'population_benefit': result['population_benefit'],
    'risk_reduction': result['risk_reduction'],
}

# Example placeholder — plug in your preferred LLM call here.
explanation = (
    'The selected portfolio prioritizes zones where high population exposure overlaps with '
    'significant flood vulnerability. The optimizer favors interventions that provide the '
    'greatest modeled benefit within the available budget.\n\n'
    'These results are decision-support estimates, not engineering recommendations.'
)
print(explanation)


## 10 — Policy Brief
Generate a government-style investment brief PDF.

In [ ]:
brief_path = generate_policy_brief(result, city_name=scenario.city['city_name'])
print('Policy brief written to:', brief_path)


## 11 — Gradio App (optional, shareable demo)
Run this cell to launch an interactive, clickable KEEPER app with a temporary public URL — great for testing and recording the demo video. Not used for the final deployed submission.

In [ ]:
sys.path.insert(0, 'app')
from app import build_app

demo = build_app()
demo.launch(share=True)


## Summary

This notebook proved the three things KEEPER needs to prove:

1. **The system can calculate urban risk** (Sections 03-04)
2. **The system can optimize limited money** (Sections 06, 08)
3. **The system can explain the result to a policymaker** (Sections 09-10)

Next: replace the synthetic dataset in `data/sample/zones.csv` with sourced data, validate intervention assumptions, and move the engine behind a FastAPI backend for the full web app.